# Semana 11 · ¿Cuánto se acumula con el tiempo?

Este cuaderno acompaña a la sesión de proyecto de la semana 11: calcular integrales numéricamente y usarlas para saber cuántas horas de noche oscura hay en un año.

**Cómo se usa.** Cada recuadro gris es una celda de código: selecciónala y pulsa `Mayús + Intro` para ejecutarla. Ejecuta las celdas **en orden**, porque las de más abajo usan variables definidas en las de arriba. Puedes cambiar cualquier número y volver a ejecutar.

Tiempo estimado: 30-45 minutos.

## 1. El área bajo una curva

Si un cuerpo va a velocidad constante $v$ durante un tiempo $t$, la distancia recorrida $v\,t$ es el **área del rectángulo** que forma la gráfica de $v(t)$ con el eje horizontal. Cuando la velocidad varía, la distancia sigue siendo el **área bajo la curva**. Ese área se llama **integral** de $v$ entre dos instantes: $\int_a^b v\,dt$.

Para calcularla numéricamente, se corta el intervalo en trozos pequeños y se suman las áreas de rectángulos o de trapecios.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def integral_izquierda(f, a, b, n):
    x = np.linspace(a, b, n + 1)
    return np.sum(f(x[:-1])) * (b - a) / n

def integral_derecha(f, a, b, n):
    x = np.linspace(a, b, n + 1)
    return np.sum(f(x[1:])) * (b - a) / n

def integral_trapecios(f, a, b, n):
    x = np.linspace(a, b, n + 1)
    y = f(x)
    return (b - a) / n * (y[0] / 2 + np.sum(y[1:-1]) + y[-1] / 2)

f = lambda x: x**2
print("Exacto de ∫₀³ x² dx = 9")
for n in [3, 10, 100, 1000]:
    print(f"n = {n:4d}   izquierda {integral_izquierda(f, 0, 3, n):8.4f}   derecha {integral_derecha(f, 0, 3, n):8.4f}   trapecios {integral_trapecios(f, 0, 3, n):8.4f}")

La suma por la izquierda se queda corta y la de la derecha se pasa (la función es creciente); la de trapecios queda en medio y converge más deprisa.

**Ejercicio.** Una piedra cae con $v(t) = 9{,}80665\,t$. Calcula con el método de los trapecios la distancia que recorre en 3 s y compárala con $\tfrac{1}{2}\,g\,t^2$.

In [ ]:
# Tu código aquí

In [ ]:
# Solución
g = 9.80665
v = lambda t: g * t
print(integral_trapecios(v, 0, 3, 100), 0.5 * g * 3**2)

## 2. La noche astronómica

El cielo se considera completamente oscuro cuando el Sol está más de 18° por debajo del horizonte (**crepúsculo astronómico**). Los crepúsculos *civil* y *náutico* corresponden a −6° y −12°. Para saber cuánto tiempo está el Sol por encima de una altura $h_0$ en un día (con la declinación de la semana 1), usamos una fórmula de astronomía de posición que todavía no puedes demostrar, así que la tratamos como caja negra:

$$\cos H = \frac{\sin h_0 - \sin\varphi\,\sin\delta}{\cos\varphi\,\cos\delta}, \qquad \text{horas por encima de } h_0 = \frac{2H}{15^\circ}.$$

In [ ]:
OBLICUIDAD = 23.44
def declinacion(d):
    return -OBLICUIDAD * np.cos(np.radians(360 / 365 * (d + 10)))

def horas_sol_por_encima(lat, d, h0):
    """Horas del día en que el Sol está por encima de la altura h0 (en grados). h0 = 0: día; h0 = -18: no hay noche astronómica."""
    lat, dec, h0 = np.radians(lat), np.radians(declinacion(d)), np.radians(h0)
    cos_h = (np.sin(h0) - np.sin(lat) * np.sin(dec)) / (np.cos(lat) * np.cos(dec))
    return 2 * np.degrees(np.arccos(np.clip(cos_h, -1, 1))) / 15

def horas_noche_astronomica(lat, d):
    return 24 - horas_sol_por_encima(lat, d, -18)

mi_latitud = 40
for nombre, d in [("solsticio de junio", 172), ("equinoccio de septiembre", 266), ("solsticio de diciembre", 355)]:
    print(f"{nombre:26s} luz: {horas_sol_por_encima(mi_latitud, d, 0):5.2f} h   noche astronómica: {horas_noche_astronomica(mi_latitud, d):5.2f} h")

## 3. Las horas de noche oscura de todo un año

Sumar las horas de noche astronómica de los 365 días es calcular el área bajo la curva diaria: una integral.

In [ ]:
dias = np.arange(1, 366)
noche = np.array([horas_noche_astronomica(mi_latitud, d) for d in dias])

plt.figure(figsize=(9, 4))
plt.plot(dias, noche)
plt.fill_between(dias, noche, alpha=0.2)
plt.xlabel("día del año"); plt.ylabel("horas de noche astronómica")
plt.title(f"Noche astronómica a lo largo del año (latitud {mi_latitud}°)")
plt.grid(alpha=0.3); plt.show()

total = integral_trapecios(lambda d: np.array([horas_noche_astronomica(mi_latitud, x) for x in d]), 1, 365, 364)
print(f"Horas de noche astronómica en un año a {mi_latitud}°: {total:,.0f} h  (de un total de {365 * 24:,} h)")

Esto es un modelo: no incluye el crepúsculo parcial por la Luna, las nubes ni la contaminación lumínica, que en una ciudad hacen que casi nunca veas un cielo realmente oscuro.

## 4. ¿Cómo cambia con la latitud?

In [ ]:
latitudes = np.arange(0, 71, 5)
totales = []
for lat in latitudes:
    n = np.array([horas_noche_astronomica(lat, d) for d in dias])
    totales.append(np.sum(n))        # suma simple de las horas diarias: cada día cuenta 1 d

plt.figure(figsize=(8, 4))
plt.bar(latitudes, totales, width=4)
plt.xlabel("latitud (°)"); plt.ylabel("horas de noche astronómica al año")
plt.title("Noche astronómica anual según la latitud")
plt.grid(alpha=0.3, axis="y"); plt.show()
print(f"En junio, a más de {90 - 18 - OBLICUIDAD:.1f}° de latitud norte el Sol no baja de -18° y no hay noche astronómica.")

## 5. Para pensar

1. ¿Cuánto vale la integral de una velocidad constante de 20 m/s entre $t = 0$ y $t = 5$ s? ¿Y la de una velocidad que crece de 0 a 20 m/s en 5 s?
2. Calcula la integral numérica de $\sin(x)$ entre 0 y $\pi$. ¿A qué valor tiende?
3. El total anual baja de unas 3460 h en el ecuador a unas 1900 h a 70° de latitud, aunque en invierno las noches son mucho más largas. ¿Por qué? (Pista: mira la curva de verano.)